# CIFAR-10 Classification: ANN vs CNN vs Hybrid Model Comparison

##Assignment: Deep Learning Assignment 2
## M Zain Ul Abideen
## 20i-0821

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))


PyTorch version: 2.8.0+cu126
CUDA available: True
GPU name: Tesla T4


In [ ]:
# Environment Setup and Imports
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import time
import os
import json
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Set seeds for reproducibility
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("CIFAR-10 Classification: Enhanced ANN vs CNN vs Hybrid Model Comparison")
print("=" * 80)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# CIFAR-10 class names
CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']

print(f"Dataset: CIFAR-10 with {len(CIFAR10_CLASSES)} classes")
print("=" * 80)


CIFAR-10 Classification: Enhanced ANN vs CNN vs Hybrid Model Comparison
PyTorch version: 2.8.0+cu126
CUDA available: True
CUDA device: Tesla T4
GPU Memory: 15.8 GB
Using device: cuda
Dataset: CIFAR-10 with 10 classes


In [ ]:
# Enhanced Data Loading with Extensive Augmentation
def get_enhanced_data_loaders(batch_size=128, num_workers=4, use_augmentation=True):
    """
    Create enhanced data loaders for CIFAR-10 with extensive augmentation
    """
    print("Loading CIFAR-10 dataset with enhanced augmentation...")

    # Base transforms
    base_transforms = [
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
    ]

    # Enhanced training transforms with extensive augmentation
    if use_augmentation:
        train_transforms = transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
            transforms.RandomRotation(15),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
            transforms.RandomPerspective(distortion_scale=0.1, p=0.3),
            *base_transforms
        ])
    else:
        train_transforms = transforms.Compose(base_transforms)

    # Test transforms (no augmentation)
    test_transforms = transforms.Compose(base_transforms)

    # Load datasets
    train_dataset = CIFAR10(root='./data', train=True, download=True, transform=train_transforms)
    test_dataset = CIFAR10(root='./data', train=False, download=True, transform=test_transforms)

    # Create data loaders with optimized settings
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                             num_workers=num_workers, pin_memory=True, persistent_workers=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                            num_workers=num_workers, pin_memory=True, persistent_workers=True)

    return train_loader, test_loader

# Create enhanced data loaders
train_loader, test_loader = get_enhanced_data_loaders(batch_size=128, use_augmentation=True)

print(f"Training samples: {len(train_loader.dataset):,}")
print(f"Test samples: {len(test_loader.dataset):,}")
print(f"Number of classes: {len(CIFAR10_CLASSES)}")
print(f"Image size: {train_loader.dataset[0][0].shape}")
print(f"Batch size: {train_loader.batch_size}")
print("Data augmentation: Random crop, flip, color jitter, rotation, perspective")
print("=" * 80)


Loading CIFAR-10 dataset with enhanced augmentation...


100%|██████████| 170M/170M [00:05<00:00, 32.7MB/s]


Training samples: 50,000
Test samples: 10,000
Number of classes: 10
Image size: torch.Size([3, 32, 32])
Batch size: 128
Data augmentation: Random crop, flip, color jitter, rotation, perspective


In [ ]:
# Enhanced Model Architectures - Target: ANN ≥75%, CNN ≥85%

class UltraDeepResidualANN(nn.Module):
    """
    Ultra-Deep Residual ANN with advanced techniques, target: ≥75% accuracy
    Enhanced with: Wider layers, residual blocks, better initialization
    """
    def __init__(self, input_dim=3072, num_classes=10, dropout_rate=0.2):
        super(UltraDeepResidualANN, self).__init__()

        # Enhanced input projection with wider layers
        self.input_proj = nn.Linear(input_dim, 2048)
        self.input_bn = nn.BatchNorm1d(2048)

        # Multiple residual blocks with increasing complexity
        self.res_blocks = nn.ModuleList([
            self._make_enhanced_residual_block(2048, 2048, dropout_rate),
            self._make_enhanced_residual_block(2048, 2048, dropout_rate),
            self._make_enhanced_residual_block(2048, 2048, dropout_rate),
            self._make_enhanced_residual_block(2048, 2048, dropout_rate),
        ])

        # Progressive reduction with wider intermediate layers
        self.reduction_layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(2048, 1024),
                nn.BatchNorm1d(1024),
                nn.GELU(),
                nn.Dropout(dropout_rate)
            ),
            nn.Sequential(
                nn.Linear(1024, 512),
                nn.BatchNorm1d(512),
                nn.GELU(),
                nn.Dropout(dropout_rate)
            ),
            nn.Sequential(
                nn.Linear(512, 256),
                nn.BatchNorm1d(256),
                nn.GELU(),
                nn.Dropout(dropout_rate)
            )
        ])

        # Output layer
        self.output_layer = nn.Linear(256, num_classes)

        # Initialize weights with advanced techniques
        self._initialize_weights()

    def _make_enhanced_residual_block(self, in_dim, out_dim, dropout_rate):
        return nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(out_dim, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(out_dim, out_dim),
            nn.BatchNorm1d(out_dim)
        )

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                # Use He initialization for better gradient flow
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        # Flatten input
        x = x.view(x.size(0), -1)

        # Enhanced input projection
        x = F.gelu(self.input_bn(self.input_proj(x)))

        # Multiple residual blocks with skip connections
        for res_block in self.res_blocks:
            residual = x
            x = res_block(x)
            x = x + residual  # Skip connection
            x = F.gelu(x)

        # Progressive reduction
        for layer in self.reduction_layers:
            x = layer(x)

        # Output
        x = self.output_layer(x)
        return x


class DeepCNN(nn.Module):
    """
    Deep CNN with residual connections, target: ≥85% accuracy
    """
    def __init__(self, num_classes=10, dropout_rate=0.3):
        super(DeepCNN, self).__init__()

        # Initial conv layer
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)

        # Conv blocks with residual connections
        self.conv_blocks = nn.ModuleList([
            self._make_conv_block(64, 128, dropout_rate),
            self._make_conv_block(128, 256, dropout_rate),
            self._make_conv_block(256, 512, dropout_rate),
            self._make_conv_block(512, 512, dropout_rate),
        ])

        # Global average pooling
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes)
        )

        # Initialize weights
        self._initialize_weights()

    def _make_conv_block(self, in_channels, out_channels, dropout_rate):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
            nn.Dropout2d(dropout_rate),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.MaxPool2d(2, 2)
        )

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        # Initial conv
        x = F.gelu(self.bn1(self.conv1(x)))
        x = F.max_pool2d(x, 2, 2)

        # Conv blocks
        for conv_block in self.conv_blocks:
            x = conv_block(x)

        # Global average pooling
        x = self.global_avg_pool(x)
        x = x.view(x.size(0), -1)

        # Classifier
        x = self.classifier(x)
        return x


class HybridCNNANN(nn.Module):
    """
    Hybrid model: CNN feature extractor + Deep ANN classifier, target: ≥85% accuracy
    """
    def __init__(self, num_classes=10, dropout_rate=0.3):
        super(HybridCNNANN, self).__init__()

        # Enhanced CNN feature extractor
        self.feature_extractor = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.GELU(),
            nn.MaxPool2d(2, 2),

            # Block 2
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.GELU(),
            nn.MaxPool2d(2, 2),

            # Block 3
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.GELU(),
            nn.MaxPool2d(2, 2),

            # Block 4
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.GELU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        # Deep ANN classifier with residual connections
        self.classifier = nn.Sequential(
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024),
            nn.GELU(),
            nn.Dropout(dropout_rate),

            nn.Linear(1024, 1024),
            nn.BatchNorm1d(1024),
            nn.GELU(),
            nn.Dropout(dropout_rate),

            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout_rate),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout_rate),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # Extract features using CNN
        features = self.feature_extractor(x)
        features = features.view(features.size(0), -1)

        # Classify using deep ANN
        output = self.classifier(features)
        return output

# Test model architectures
print("Testing model architectures...")

# Test input
test_input = torch.randn(2, 3, 32, 32).to(device)

# Test Enhanced ANN
enhanced_ann_model = UltraDeepResidualANN().to(device)
enhanced_ann_output = enhanced_ann_model(test_input)
print(f"Enhanced ANN - Input: {test_input.shape}, Output: {enhanced_ann_output.shape}")
print(f"Enhanced ANN - Parameters: {sum(p.numel() for p in enhanced_ann_model.parameters()):,}")

# Test CNN
cnn_model = DeepCNN().to(device)
cnn_output = cnn_model(test_input)
print(f"CNN - Input: {test_input.shape}, Output: {cnn_output.shape}")
print(f"CNN - Parameters: {sum(p.numel() for p in cnn_model.parameters()):,}")

# Test Hybrid
hybrid_model = HybridCNNANN().to(device)
hybrid_output = hybrid_model(test_input)
print(f"Hybrid - Input: {test_input.shape}, Output: {hybrid_output.shape}")
print(f"Hybrid - Parameters: {sum(p.numel() for p in hybrid_model.parameters()):,}")

print("All model architectures tested successfully!")
print("=" * 80)


Testing model architectures...
Enhanced ANN - Input: torch.Size([2, 3, 32, 32]), Output: torch.Size([2, 10])
Enhanced ANN - Parameters: 59,463,434
CNN - Input: torch.Size([2, 3, 32, 32]), Output: torch.Size([2, 10])
CNN - Parameters: 9,771,914
Hybrid - Input: torch.Size([2, 3, 32, 32]), Output: torch.Size([2, 10])
Hybrid - Parameters: 3,792,138
All model architectures tested successfully!


In [ ]:
# Enhanced Training Utilities - Optimized for high accuracy

class EarlyStopping:
    def __init__(self, patience=15, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, val_score):
        if self.best_score is None:
            self.best_score = val_score
        elif val_score < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = val_score
            self.counter = 0

def train_enhanced_model(model, train_loader, test_loader, epochs=50, lr=0.001,
                        weight_decay=1e-4, model_name='Model', device='cuda'):
    """
    Enhanced training function with advanced optimization techniques
    """
    model = model.to(device)

    # Enhanced loss function with label smoothing
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    # Advanced optimizer with different settings for different model types
    if 'ANN' in model_name:
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay, betas=(0.9, 0.999))
    else:  # CNN or Hybrid
        optimizer = optim.AdamW(model.parameters(), lr=lr*0.5, weight_decay=weight_decay*2, betas=(0.9, 0.999))

    # Learning rate scheduler with warmup
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=lr*0.01)

    # Early stopping
    early_stopping = EarlyStopping(patience=20, min_delta=0.001)

    # Training history
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
        'lr': []
    }

    best_val_acc = 0.0
    best_model_state = None

    print(f"Training {model_name} for {epochs} epochs...")
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Initial LR: {lr}, Weight Decay: {weight_decay}")
    print(f"Target: {'≥75%' if 'ANN' in model_name else '≥85%'} accuracy")
    print("-" * 70)

    start_time = time.time()

    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            train_loss += loss.item()
            _, predicted = output.max(1)
            train_total += target.size(0)
            train_correct += predicted.eq(target).sum().item()

        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                loss = criterion(output, target)

                val_loss += loss.item()
                _, predicted = output.max(1)
                val_total += target.size(0)
                val_correct += predicted.eq(target).sum().item()

        # Calculate metrics
        train_loss /= len(train_loader)
        train_acc = 100. * train_correct / train_total
        val_loss /= len(test_loader)
        val_acc = 100. * val_correct / val_total

        # Update scheduler
        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]

        # Store history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['lr'].append(current_lr)

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()

        # Print progress
        if (epoch + 1) % 10 == 0 or epoch == 0:
            elapsed = time.time() - start_time
            print(f"Epoch [{epoch+1:3d}/{epochs}] - "
                  f"Train: {train_acc:5.2f}% | Val: {val_acc:5.2f}% | "
                  f"LR: {current_lr:.6f} | Time: {elapsed/60:.1f}m")

        # Early stopping check
        early_stopping(val_acc)
        if early_stopping.early_stop:
            print(f"Early stopping at epoch {epoch+1}")
            break

    # Load best model
    if best_model_state:
        model.load_state_dict(best_model_state)

    total_time = time.time() - start_time
    print(f"Best validation accuracy: {best_val_acc:.2f}%")
    print(f"Total training time: {total_time/60:.1f} minutes")

    return model, history, best_val_acc

def evaluate_model(model, test_loader, class_names=None, device='cuda'):
    """
    Enhanced evaluation function with detailed metrics
    """
    model.eval()
    all_predictions = []
    all_targets = []
    test_loss = 0.0
    correct = 0
    total = 0

    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)

            test_loss += loss.item()
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()

            all_predictions.extend(predicted.cpu().numpy())
            all_targets.extend(target.cpu().numpy())

    test_loss /= len(test_loader)
    accuracy = 100. * correct / total

    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {accuracy:.2f}%")

    return accuracy, all_predictions, all_targets

print("Enhanced training utilities defined successfully!")
print("=" * 80)


Enhanced training utilities defined successfully!


In [34]:
# Train Enhanced ANN (Target: ≥75% accuracy)

print("\nTRAINING ENHANCED ULTRA-DEEP RESIDUAL ANN")
print("=" * 60)

# Train Enhanced ANN
enhanced_ann_trained, enhanced_ann_history, enhanced_ann_best_val = train_enhanced_model(
    enhanced_ann_model, train_loader, test_loader,
    epochs=120,
    lr=0.005,
    weight_decay=1e-4,
    model_name='Enhanced Ultra-Deep ANN',
    device=device
)

# Evaluate Enhanced ANN
print("\nEVALUATING ENHANCED ULTRA-DEEP RESIDUAL ANN")
print("=" * 60)
enhanced_ann_test_acc, enhanced_ann_preds, enhanced_ann_targets = evaluate_model(
    enhanced_ann_trained, test_loader, CIFAR10_CLASSES, device
)

results['Enhanced Ultra-Deep ANN'] = {
    'test_accuracy': enhanced_ann_test_acc,
    'val_accuracy': enhanced_ann_best_val,
    'parameters': sum(p.numel() for p in enhanced_ann_trained.parameters()),
    'history': enhanced_ann_history
}

print(f"\nEnhanced ANN Training Complete!")
print(f"Final Test Accuracy: {enhanced_ann_test_acc:.2f}%")
print(f"Model Parameters: {results['Enhanced Ultra-Deep ANN']['parameters']:,}")

# Check if target achieved
ann_target_achieved = enhanced_ann_test_acc >= 75
print(f"Target Achievement: {'ACHIEVED' if ann_target_achieved else 'NOT ACHIEVED'} (Target: ≥75%)")
print("=" * 80)



TRAINING ENHANCED ULTRA-DEEP RESIDUAL ANN
Training Enhanced Ultra-Deep ANN for 120 epochs...
Model parameters: 59,463,434
Initial LR: 0.005, Weight Decay: 0.0001
Target: ≥75% accuracy
----------------------------------------------------------------------
Epoch [  1/120] - Train: 50.67% | Val: 55.60% | LR: 0.004879 | Time: 0.9m
Epoch [ 10/120] - Train: 57.76% | Val: 62.83% | LR: 0.005000 | Time: 8.8m
Epoch [ 20/120] - Train: 54.99% | Val: 61.63% | LR: 0.002525 | Time: 17.5m
Epoch [ 30/120] - Train: 60.31% | Val: 64.59% | LR: 0.005000 | Time: 26.3m
Epoch [ 40/120] - Train: 55.57% | Val: 60.89% | LR: 0.004275 | Time: 35.1m
Epoch [ 50/120] - Train: 58.99% | Val: 63.51% | LR: 0.002525 | Time: 44.0m
Early stopping at epoch 50
Best validation accuracy: 64.59%
Total training time: 44.0 minutes

EVALUATING ENHANCED ULTRA-DEEP RESIDUAL ANN
Test Loss: 1.0711
Test Accuracy: 63.51%

Enhanced ANN Training Complete!
Final Test Accuracy: 63.51%
Model Parameters: 59,463,434
Target Achievement: NOT ACH

In [8]:
# Train Deep CNN (Target: ≥85% accuracy)

print("\nTRAINING DEEP CNN")
print("=" * 60)

# Train CNN
cnn_trained, cnn_history, cnn_best_val = train_enhanced_model(
    cnn_model, train_loader, test_loader,
    epochs=50,
    lr=0.001,
    weight_decay=1e-4,
    model_name='Deep CNN',
    device=device
)

# Evaluate CNN
print("\nEVALUATING DEEP CNN")
print("=" * 60)
cnn_test_acc, cnn_preds, cnn_targets = evaluate_model(
    cnn_trained, test_loader, CIFAR10_CLASSES, device
)

results['Deep CNN'] = {
    'test_accuracy': cnn_test_acc,
    'val_accuracy': cnn_best_val,
    'parameters': sum(p.numel() for p in cnn_trained.parameters()),
    'history': cnn_history
}

print(f"\nCNN Training Complete!")
print(f"Final Test Accuracy: {cnn_test_acc:.2f}%")
print(f"Model Parameters: {results['Deep CNN']['parameters']:,}")

# Check if target achieved
cnn_target_achieved = cnn_test_acc >= 85
print(f"Target Achievement: {'ACHIEVED' if cnn_target_achieved else 'NOT ACHIEVED'} ")
print("=" * 80)



TRAINING DEEP CNN
Training Deep CNN for 50 epochs...
Model parameters: 9,771,914
Initial LR: 0.001, Weight Decay: 0.0001
Target: ≥85% accuracy
----------------------------------------------------------------------
Epoch [  1/50] - Train: 29.77% | Val: 46.09% | LR: 0.000488 | Time: 0.9m
Epoch [ 10/50] - Train: 68.37% | Val: 77.55% | LR: 0.000500 | Time: 8.9m
Epoch [ 20/50] - Train: 74.93% | Val: 83.13% | LR: 0.000255 | Time: 17.7m
Epoch [ 30/50] - Train: 79.88% | Val: 86.29% | LR: 0.000500 | Time: 26.4m
Epoch [ 40/50] - Train: 79.11% | Val: 85.83% | LR: 0.000428 | Time: 35.1m
Epoch [ 50/50] - Train: 82.91% | Val: 88.56% | LR: 0.000255 | Time: 43.8m
Best validation accuracy: 88.56%
Total training time: 43.8 minutes

EVALUATING DEEP CNN
Test Loss: 0.4079
Test Accuracy: 88.56%

CNN Training Complete!
Final Test Accuracy: 88.56%
Model Parameters: 9,771,914
Target Achievement: ACHIEVED 


In [9]:
# Train Hybrid CNN+ANN (Target: ≥85% accuracy)

print("\nTRAINING HYBRID CNN+ANN")
print("=" * 60)

# Train Hybrid
hybrid_trained, hybrid_history, hybrid_best_val = train_enhanced_model(
    hybrid_model, train_loader, test_loader,
    epochs=50,
    lr=0.001,
    weight_decay=1e-4,
    model_name='Hybrid CNN+ANN',
    device=device
)

# Evaluate Hybrid
print("\nEVALUATING HYBRID CNN+ANN")
print("=" * 60)
hybrid_test_acc, hybrid_preds, hybrid_targets = evaluate_model(
    hybrid_trained, test_loader, CIFAR10_CLASSES, device
)

results['Hybrid CNN+ANN'] = {
    'test_accuracy': hybrid_test_acc,
    'val_accuracy': hybrid_best_val,
    'parameters': sum(p.numel() for p in hybrid_trained.parameters()),
    'history': hybrid_history
}

print(f"\nHybrid Training Complete!")
print(f"Final Test Accuracy: {hybrid_test_acc:.2f}%")
print(f"Model Parameters: {results['Hybrid CNN+ANN']['parameters']:,}")

# Check if target achieved
hybrid_target_achieved = hybrid_test_acc >= 85
print(f"Target Achievement: {'ACHIEVED' if hybrid_target_achieved else 'NOT ACHIEVED'} (Target: ≥85%)")
print("=" * 80)



TRAINING HYBRID CNN+ANN
Training Hybrid CNN+ANN for 50 epochs...
Model parameters: 3,792,138
Initial LR: 0.001, Weight Decay: 0.0001
Target: ≥75% accuracy
----------------------------------------------------------------------
Epoch [  1/50] - Train: 36.34% | Val: 52.25% | LR: 0.000976 | Time: 0.8m
Epoch [ 10/50] - Train: 75.45% | Val: 80.05% | LR: 0.001000 | Time: 8.5m
Epoch [ 20/50] - Train: 79.37% | Val: 83.01% | LR: 0.000505 | Time: 17.0m
Epoch [ 30/50] - Train: 84.92% | Val: 86.93% | LR: 0.001000 | Time: 25.6m
Epoch [ 40/50] - Train: 83.73% | Val: 85.75% | LR: 0.000855 | Time: 34.2m
Epoch [ 50/50] - Train: 87.72% | Val: 87.23% | LR: 0.000505 | Time: 42.6m
Best validation accuracy: 87.67%
Total training time: 42.6 minutes

EVALUATING HYBRID CNN+ANN
Test Loss: 0.4336
Test Accuracy: 87.23%

Hybrid Training Complete!
Final Test Accuracy: 87.23%
Model Parameters: 3,792,138
Target Achievement: ACHIEVED (Target: ≥85%)


In [ ]:
# Comprehensive Results Comparison and Analysis


In [ ]:
# Comprehensive Results Comparison and Analysis

print("\n" + "="*80)
print("COMPREHENSIVE RESULTS COMPARISON")
print("="*80)

# Create comparison table
comparison_data = []
for model_name, result in results.items():
    comparison_data.append({
        'Model': model_name,
        'Test Accuracy (%)': f"{result['test_accuracy']:.2f}",
        'Val Accuracy (%)': f"{result['val_accuracy']:.2f}",
        'Parameters': f"{result['parameters']:,}",
        'Target Met': 'YES' if (result['test_accuracy'] >= 70 if 'ANN' in model_name else result['test_accuracy'] >= 75) else 'NO'
    })

comparison_df = pd.DataFrame(comparison_data)
print("\nFINAL RESULTS TABLE:")
print(comparison_df.to_string(index=False))

# Summary statistics
print("\nSUMMARY STATISTICS:")
print(f"Best Model: {max(results.items(), key=lambda x: x[1]['test_accuracy'])[0]}")
print(f"Best Accuracy: {max(results.values(), key=lambda x: x['test_accuracy'])['test_accuracy']:.2f}%")
print(f"Average Accuracy: {np.mean([r['test_accuracy'] for r in results.values()]):.2f}%")

# Target achievement check
enhanced_ann_acc = results['Enhanced Ultra-Deep ANN']['test_accuracy']
cnn_acc = results['Deep CNN']['test_accuracy']
hybrid_acc = results['Hybrid CNN+ANN']['test_accuracy']

print("\nTARGET ACHIEVEMENT:")
print(f" ANN Target : {'ACHIEVED' if enhanced_ann_acc >= 75 else 'NOT ACHIEVED'} ({enhanced_ann_acc:.2f}%)")
print(f"CNN Target : {'ACHIEVED' if cnn_acc >= 85 else 'NOT ACHIEVED'} ({cnn_acc:.2f}%)")
print(f"Hybrid Target : {'ACHIEVED' if hybrid_acc >= 85 else 'NOT ACHIEVED'} ({hybrid_acc:.2f}%)")

# Overall success
overall_success = enhanced_ann_acc >= 75 and cnn_acc >= 85 and hybrid_acc >= 85
print(f"\nOVERALL SUCCESS: {'ALL TARGETS ACHIEVED!' if overall_success else 'SOME TARGETS NOT MET'}")

# Performance comparison
print("\nPERFORMANCE COMPARISON:")
print(f" ANN vs CNN: {enhanced_ann_acc:.2f}% vs {cnn_acc:.2f}% (Difference: {abs(enhanced_ann_acc - cnn_acc):.2f}%)")
print(f"CNN vs Hybrid: {cnn_acc:.2f}% vs {hybrid_acc:.2f}% (Difference: {abs(cnn_acc - hybrid_acc):.2f}%)")
print(f" ANN vs Hybrid: {enhanced_ann_acc:.2f}% vs {hybrid_acc:.2f}% (Difference: {abs(enhanced_ann_acc - hybrid_acc):.2f}%)")

print("\n" + "="*80)
print("TRAINING COMPLETED SUCCESSFULLY!")
print("="*80)



COMPREHENSIVE RESULTS COMPARISON

FINAL RESULTS TABLE:
                  Model Test Accuracy (%) Val Accuracy (%) Parameters Target Met
Enhanced Ultra-Deep ANN             55.28            55.45 59,463,434         NO
               Deep CNN             88.56            88.56  9,771,914        YES
         Hybrid CNN+ANN             87.23            87.67  3,792,138        YES

SUMMARY STATISTICS:
Best Model: Deep CNN
Best Accuracy: 88.56%
Average Accuracy: 77.02%

TARGET ACHIEVEMENT:
 ANN Target : NOT ACHIEVED (55.28%)
CNN Target : ACHIEVED (88.56%)
Hybrid Target : ACHIEVED (87.23%)

OVERALL SUCCESS: SOME TARGETS NOT MET

PERFORMANCE COMPARISON:
 ANN vs CNN: 55.28% vs 88.56% (Difference: 33.28%)
CNN vs Hybrid: 88.56% vs 87.23% (Difference: 1.33%)
 ANN vs Hybrid: 55.28% vs 87.23% (Difference: 31.95%)

TRAINING COMPLETED SUCCESSFULLY!


In [37]:
# Comprehensive Results Comparison and Analysis

print("\n" + "="*80)
print("COMPREHENSIVE RESULTS COMPARISON")
print("="*80)

# Create comparison table
comparison_data = []
for model_name, result in results.items():
    comparison_data.append({
        'Model': model_name,
        'Test Accuracy (%)': f"{result['test_accuracy']:.2f}",
        'Val Accuracy (%)': f"{result['val_accuracy']:.2f}",
        'Parameters': f"{result['parameters']:,}",
        'Target Met': 'YES' if (result['test_accuracy'] >= 70 if 'ANN' in model_name else result['test_accuracy'] >= 75) else 'NO'
    })

comparison_df = pd.DataFrame(comparison_data)
print("\nFINAL RESULTS TABLE:")
print(comparison_df.to_string(index=False))

# Summary statistics
print("\nSUMMARY STATISTICS:")
print(f"Best Model: {max(results.items(), key=lambda x: x[1]['test_accuracy'])[0]}")
print(f"Best Accuracy: {max(results.values(), key=lambda x: x['test_accuracy'])['test_accuracy']:.2f}%")
print(f"Average Accuracy: {np.mean([r['test_accuracy'] for r in results.values()]):.2f}%")





COMPREHENSIVE RESULTS COMPARISON

FINAL RESULTS TABLE:
                  Model Test Accuracy (%) Val Accuracy (%) Parameters Target Met
Enhanced Ultra-Deep ANN             63.51            64.59 59,463,434         NO

SUMMARY STATISTICS:
Best Model: Enhanced Ultra-Deep ANN
Best Accuracy: 63.51%
Average Accuracy: 63.51%


In [25]:
# Model Checkpoint Loading and Evaluation

def load_model_checkpoint(checkpoint_path, model_class, device='cuda'):
    """Load model from checkpoint"""
    checkpoint = torch.load(checkpoint_path, map_location=device)

    # Create model instance
    model = model_class().to(device)

    # Load state dict
    model.load_state_dict(checkpoint['model_state_dict'])

    print(f"Loaded model: {checkpoint['model_name']}")
    print(f"Architecture: {checkpoint['model_architecture']}")
    print(f"Test Accuracy: {checkpoint['test_accuracy']:.2f}%")
    print(f"Parameters: {checkpoint['parameters']:,}")
    print(f"Timestamp: {checkpoint['timestamp']}")

    return model, checkpoint

def evaluate_checkpoint(checkpoint_path, model_class, test_loader, device='cuda'):
    """Load and evaluate a model from checkpoint"""
    model, checkpoint_info = load_model_checkpoint(checkpoint_path, model_class, device)

    # Evaluate the model
    accuracy, predictions, targets = evaluate_model(model, test_loader, CIFAR10_CLASSES, device)

    return model, accuracy, predictions, targets, checkpoint_info

# Example usage (uncomment to use):
# Load and evaluate a specific checkpoint
# ann_model, ann_acc, ann_preds, ann_targets, ann_info = evaluate_checkpoint(
#     './checkpoints/Enhanced_Ultra_Deep_ANN_20241224_143022.pth',
#     UltraDeepResidualANN,
#     test_loader,
#     device
# )

print("Checkpoint loading functions defined successfully!")
print("Use load_model_checkpoint() or evaluate_checkpoint() to load saved models")
print("=" * 60)


Checkpoint loading functions defined successfully!
Use load_model_checkpoint() or evaluate_checkpoint() to load saved models


In [27]:
# Complete Checkpoint Management System

import os
import json
from datetime import datetime

class CheckpointManager:
    """Complete checkpoint management system for model saving and loading"""

    def __init__(self, checkpoint_dir='./checkpoints'):
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(checkpoint_dir, exist_ok=True)
        self.timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    def save_model_checkpoint(self, model, model_name, accuracy, additional_info=None):
        """Save model checkpoint with comprehensive metadata"""
        checkpoint_path = os.path.join(self.checkpoint_dir, f"{model_name}_{self.timestamp}.pth")

        checkpoint = {
            'model_state_dict': model.state_dict(),
            'model_name': model_name,
            'test_accuracy': accuracy,
            'timestamp': self.timestamp,
            'model_architecture': str(model.__class__.__name__),
            'parameters': sum(p.numel() for p in model.parameters()),
            'device': str(next(model.parameters()).device),
            'additional_info': additional_info or {}
        }

        torch.save(checkpoint, checkpoint_path)
        print(f"✅ Checkpoint saved: {checkpoint_path}")
        print(f"   Model: {model_name}")
        print(f"   Accuracy: {accuracy:.2f}%")
        print(f"   Parameters: {checkpoint['parameters']:,}")
        return checkpoint_path

    def load_model_checkpoint(self, checkpoint_path, model_class, device='cuda'):
        """Load model from checkpoint with full metadata"""
        checkpoint = torch.load(checkpoint_path, map_location=device)

        # Create model instance
        model = model_class().to(device)
        model.load_state_dict(checkpoint['model_state_dict'])

        print(f"✅ Model loaded: {checkpoint['model_name']}")
        print(f"   Architecture: {checkpoint['model_architecture']}")
        print(f"   Test Accuracy: {checkpoint['test_accuracy']:.2f}%")
        print(f"   Parameters: {checkpoint['parameters']:,}")
        print(f"   Timestamp: {checkpoint['timestamp']}")

        return model, checkpoint

    def evaluate_checkpoint(self, checkpoint_path, model_class, test_loader, device='cuda'):
        """Load and evaluate model from checkpoint"""
        model, checkpoint_info = self.load_model_checkpoint(checkpoint_path, model_class, device)

        print(f"\n🔍 Evaluating {checkpoint_info['model_name']}...")
        accuracy, predictions, targets = evaluate_model(model, test_loader, CIFAR10_CLASSES, device)

        return model, accuracy, predictions, targets, checkpoint_info

    def save_training_summary(self, results):
        """Save comprehensive training summary"""
        summary = {
            'timestamp': self.timestamp,
            'experiment_info': {
                'dataset': 'CIFAR-10',
                'total_models': len(results),
                'target_accuracies': {
                    'ANN': '≥75%',
                    'CNN': '≥85%',
                    'Hybrid': '≥85%'
                }
            },
            'model_results': {},
            'overall_summary': {}
        }

        # Add individual model results
        for model_name, result in results.items():
            summary['model_results'][model_name] = {
                'test_accuracy': result['test_accuracy'],
                'val_accuracy': result['val_accuracy'],
                'parameters': result['parameters'],
                'target_met': result['test_accuracy'] >= (75 if 'ANN' in model_name else 85)
            }

        # Calculate overall statistics
        accuracies = [r['test_accuracy'] for r in results.values()]
        summary['overall_summary'] = {
            'best_accuracy': max(accuracies),
            'average_accuracy': sum(accuracies) / len(accuracies),
            'all_targets_met': all([
                results['Enhanced Ultra-Deep ANN']['test_accuracy'] >= 75 if 'Enhanced Ultra-Deep ANN' in results else False,
                results['Deep CNN']['test_accuracy'] >= 85 if 'Deep CNN' in results else False,
                results['Hybrid CNN+ANN']['test_accuracy'] >= 85 if 'Hybrid CNN+ANN' in results else False
            ])
        }

        summary_path = os.path.join(self.checkpoint_dir, f"training_summary_{self.timestamp}.json")
        with open(summary_path, 'w') as f:
            json.dump(summary, f, indent=2)

        print(f"✅ Training summary saved: {summary_path}")
        return summary_path

# Initialize checkpoint manager
checkpoint_manager = CheckpointManager()

print(" Checkpoint Management System Initialized")
print(f" Checkpoint directory: {checkpoint_manager.checkpoint_dir}")
print(f" Timestamp: {checkpoint_manager.timestamp}")
print("=" * 80)


 Checkpoint Management System Initialized
 Checkpoint directory: ./checkpoints
 Timestamp: 20251024_123513


In [31]:
# Save All Model Checkpoints

print(" SAVING ALL MODEL CHECKPOINTS")
print("=" * 80)

# Save Enhanced ANN checkpoint
if 'Enhanced Ultra-Deep ANN' in results:
    ann_checkpoint_path = checkpoint_manager.save_model_checkpoint(
        enhanced_ann_trained,
        'Enhanced_Ultra_Deep_ANN',
        results['Enhanced Ultra-Deep ANN']['test_accuracy'],
        {'target_accuracy': 75, 'achieved': results['Enhanced Ultra-Deep ANN']['test_accuracy'] >= 75}
    )

# Save Deep CNN checkpoint
if 'Deep CNN' in results:
    cnn_checkpoint_path = checkpoint_manager.save_model_checkpoint(
        cnn_trained,
        'Deep_CNN',
        results['Deep CNN']['test_accuracy'],
        {'target_accuracy': 85, 'achieved': results['Deep CNN']['test_accuracy'] >= 85}
    )

# Save Hybrid CNN+ANN checkpoint
if 'Hybrid CNN+ANN' in results:
    hybrid_checkpoint_path = checkpoint_manager.save_model_checkpoint(
        hybrid_trained,
        'Hybrid_CNN_ANN',
        results['Hybrid CNN+ANN']['test_accuracy'],
        {'target_accuracy': 85, 'achieved': results['Hybrid CNN+ANN']['test_accuracy'] >= 85}
    )

# Save comprehensive training summary
summary_path = checkpoint_manager.save_training_summary(results)

print("\n CHECKPOINT SUMMARY:")
print("=" * 60)
print(f" Checkpoint Directory: {checkpoint_manager.checkpoint_dir}")
print(f" Timestamp: {checkpoint_manager.timestamp}")
print(f" Summary File: {summary_path}")

# List all saved files
import glob
checkpoint_files = glob.glob(os.path.join(checkpoint_manager.checkpoint_dir, "*.pth"))
print(f"\n Saved Model Files ({len(checkpoint_files)}):")
for file in checkpoint_files:
    filename = os.path.basename(file)
    file_size = os.path.getsize(file) / (1024 * 1024)  # MB
    print(f"    {filename} ({file_size:.1f} MB)")

print("\n All model checkpoints saved successfully!")
print(" Assignment requirement 'Model checkpoint(s): include final trained weights' - COMPLETED")
print("=" * 80)


 SAVING ALL MODEL CHECKPOINTS
✅ Checkpoint saved: ./checkpoints/Enhanced_Ultra_Deep_ANN_20251024_123513.pth
   Model: Enhanced_Ultra_Deep_ANN
   Accuracy: 55.28%
   Parameters: 59,463,434
✅ Checkpoint saved: ./checkpoints/Deep_CNN_20251024_123513.pth
   Model: Deep_CNN
   Accuracy: 88.56%
   Parameters: 9,771,914
✅ Checkpoint saved: ./checkpoints/Hybrid_CNN_ANN_20251024_123513.pth
   Model: Hybrid_CNN_ANN
   Accuracy: 87.23%
   Parameters: 3,792,138
✅ Training summary saved: ./checkpoints/training_summary_20251024_123513.json

 CHECKPOINT SUMMARY:
 Checkpoint Directory: ./checkpoints
 Timestamp: 20251024_123513
 Summary File: ./checkpoints/training_summary_20251024_123513.json

 Saved Model Files (3):
    Hybrid_CNN_ANN_20251024_123513.pth (14.5 MB)
    Deep_CNN_20251024_123513.pth (37.3 MB)
    Enhanced_Ultra_Deep_ANN_20251024_123513.pth (227.1 MB)

 All model checkpoints saved successfully!
 Assignment requirement 'Model checkpoint(s): include final trained weights' - COMPLETED
